# §1.5 リーマン計量 - Fisher情報行列が計量になる

リーマン計量は多様体上で「距離」や「角度」を測る道具です。
統計的多様体では、**Fisher情報行列がリーマン計量の役割を果たします**。

## 既知概念との対応

| 情報幾何の概念 | 既知概念 |
|--------------|--------|
| リーマン計量テンソル | Fisher情報行列 |
| リーマン距離の2乗 | KLダイバージェンス（2次近似）|
| 計量の定義 | スコア関数の共分散 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
def gaussian_pdf(x, mu, sigma):
    """正規分布の確率密度関数"""
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def compute_fisher_gaussian(mu, sigma):
    """
    正規分布 N(μ, σ²) のFisher情報行列
    I = [[1/σ², 0], [0, 2/σ²]]
    """
    return np.array([[1/sigma**2, 0], [0, 2/sigma**2]])

def kl_divergence_gaussian(mu1, sigma1, mu2, sigma2):
    """D_KL(N(μ1,σ1²) || N(μ2,σ2²))"""
    return np.log(sigma2/sigma1) + (sigma1**2 + (mu1-mu2)**2)/(2*sigma2**2) - 0.5

---
## 1. Fisher情報行列の計算と意味

### 正規分布のFisher情報行列

$$I(\mu, \sigma) = \begin{pmatrix} 1/\sigma^2 & 0 \\ 0 & 2/\sigma^2 \end{pmatrix}$$

### 各成分の意味
- $I_{\mu\mu} = 1/\sigma^2$: μに関する情報量（σが小さいほど大）
- $I_{\sigma\sigma} = 2/\sigma^2$: σに関する情報量
- $I_{\mu\sigma} = 0$: μとσは統計的に独立

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sigma_range = np.linspace(0.3, 3, 100)

# I_μμ = 1/σ²
ax1 = axes[0]
I_mu_mu = 1 / sigma_range**2
ax1.plot(sigma_range, I_mu_mu, 'r-', linewidth=2)
ax1.set_xlabel('σ', fontsize=12)
ax1.set_ylabel('I_μμ = 1/σ²', fontsize=12)
ax1.set_title('Fisher information for μ\nSmaller σ → more information')
ax1.grid(True, alpha=0.3)

# I_σσ = 2/σ²
ax2 = axes[1]
I_sigma_sigma = 2 / sigma_range**2
ax2.plot(sigma_range, I_sigma_sigma, 'b-', linewidth=2)
ax2.set_xlabel('σ', fontsize=12)
ax2.set_ylabel('I_σσ = 2/σ²', fontsize=12)
ax2.set_title('Fisher information for σ')
ax2.grid(True, alpha=0.3)

# 情報行列の楕円表示
ax3 = axes[2]
theta = np.linspace(0, 2*np.pi, 100)
points = [(0, 0.5, 'red'), (0, 1.0, 'blue'), (0, 2.0, 'green')]

for mu, sigma, color in points:
    # 楕円: dμ²/σ² + 2dσ²/σ² = const
    # 半径を σ に比例させる（見やすさのため）
    scale = 0.15
    ellipse_x = sigma * np.cos(theta) * scale + mu
    ellipse_y = sigma/np.sqrt(2) * np.sin(theta) * scale + sigma
    ax3.plot(ellipse_x, ellipse_y, color=color, linewidth=2, label=f'σ={sigma}')
    ax3.plot(mu, sigma, 'o', color=color, markersize=8)

ax3.set_xlabel('μ', fontsize=12)
ax3.set_ylabel('σ', fontsize=12)
ax3.set_title('Fisher metric ellipses\nSmaller σ → smaller ellipse')
ax3.legend()
ax3.set_xlim(-0.5, 0.5)
ax3.set_ylim(0, 2.5)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 学習ポイント

1. **Fisher情報行列** $I(\theta) = E[(\partial\log p/\partial\theta)(\partial\log p/\partial\theta)^\top]$
2. 正規分布では $I = \text{diag}(1/\sigma^2, 2/\sigma^2)$
3. **σが小さい → Fisher情報が大きい → 推定精度が高い**
4. Cramér-Raoの下界: $\text{Var}(\hat{\theta}) \geq I(\theta)^{-1}$
5. 楕円が小さい ⇔ 推定誤差の下界が小さい

---
## 2. リーマン計量としてのFisher情報

### リーマン計量の役割

- 接ベクトルの「長さ」を測る: $||v||^2 = v^\top I v$
- 接ベクトル間の「角度」を測る: $\cos\theta = \langle u,v \rangle / (||u|| ||v||)$

### 重要な性質

Fisher情報行列 $I(\theta)$ は：
1. **対称**: $I_{ij} = I_{ji}$
2. **正定値**: $v^\top I v > 0$ for all $v \neq 0$
3. → **リーマン計量の条件を満たす**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mu0, sigma0 = 0, 1
theta = np.linspace(0, 2*np.pi, 100)

# 左図：ユークリッド計量 vs Fisher計量
ax1 = axes[0]

# 単位円（ユークリッド）
x_euclid = np.cos(theta) * 0.5
y_euclid = np.sin(theta) * 0.5
ax1.plot(x_euclid + mu0, y_euclid + sigma0, 'b--', linewidth=2, label='Euclidean unit circle')

# Fisher計量による「単位円」（楕円になる）
x_fisher = sigma0 * np.cos(theta) * 0.5
y_fisher = sigma0 / np.sqrt(2) * np.sin(theta) * 0.5
ax1.plot(x_fisher + mu0, y_fisher + sigma0, 'r-', linewidth=2, label='Fisher unit "circle"')

ax1.plot(mu0, sigma0, 'ko', markersize=10)
ax1.text(mu0 - 0.1, sigma0 - 0.15, 'p', fontsize=12, fontweight='bold')

ax1.set_xlabel('μ', fontsize=12)
ax1.set_ylabel('σ', fontsize=12)
ax1.set_title('Euclidean vs Fisher metric\n"Unit ball" shapes differ', fontsize=11)
ax1.legend()
ax1.set_aspect('equal')
ax1.grid(True, alpha=0.3)

# 右図：異なる点での計量楕円
ax2 = axes[1]

points = [
    (0, 0.5, 'red'),
    (1, 1.0, 'blue'),
    (-1, 1.5, 'green'),
    (0, 2.0, 'purple'),
]

for mu, sigma, color in points:
    x = sigma * np.cos(theta) * 0.2
    y = sigma / np.sqrt(2) * np.sin(theta) * 0.2
    ax2.plot(x + mu, y + sigma, color=color, linewidth=1.5)
    ax2.plot(mu, sigma, 'o', color=color, markersize=8)

ax2.set_xlabel('μ', fontsize=12)
ax2.set_ylabel('σ', fontsize=12)
ax2.set_title('Fisher metric ellipses at different points\nSmaller σ → smaller ellipse (feels "farther")', fontsize=11)
ax2.set_xlim(-2, 2)
ax2.set_ylim(0, 3)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 学習ポイント

1. リーマン計量 $g(v, w)$ は各点で定義される内積
2. **Fisher情報行列 $I(\theta)$ がこの役割を果たす**
3. $||v||^2_{\text{Fisher}} = v^\top I(\theta) v$
4. σが小さい領域では「距離が大きく感じる」
   - 狭い分布を少し変えると大きな情報変化
5. **これが「情報幾何」の名前の由来**

---
## 3. KLダイバージェンスとの関係

### 核心的な等式

KLダイバージェンスの2次近似がリーマン距離の2乗になる：

$$D_{\text{KL}}(p_\theta \| p_{\theta+d\theta}) \approx \frac{1}{2} d\theta^\top I(\theta) d\theta$$

**これが「Fisher情報 = リーマン計量」の根拠**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mu0, sigma0 = 0, 1

mu_range = np.linspace(-1, 1, 100)
sigma_range = np.linspace(0.5, 1.5, 100)
MU, SIGMA = np.meshgrid(mu_range, sigma_range)

# KLダイバージェンス D_KL(N(μ₀,σ₀²) || N(μ,σ²))
KL = np.log(SIGMA/sigma0) + (sigma0**2 + (mu0-MU)**2)/(2*SIGMA**2) - 0.5

# 2次近似 (1/2)(dμ²/σ₀² + 2dσ²/σ₀²)
dMU = MU - mu0
dSIGMA = SIGMA - sigma0
QUAD_APPROX = 0.5 * (dMU**2 / sigma0**2 + 2 * dSIGMA**2 / sigma0**2)

levels = [0.01, 0.05, 0.1, 0.2, 0.5]

# 左図：真のKL等高線
ax1 = axes[0]
cs1 = ax1.contour(MU, SIGMA, KL, levels=levels, colors='blue')
ax1.clabel(cs1, inline=True, fontsize=8)
ax1.plot(mu0, sigma0, 'r*', markersize=15, label='Reference point')
ax1.set_xlabel('μ', fontsize=12)
ax1.set_ylabel('σ', fontsize=12)
ax1.set_title('True KL divergence contours', fontsize=11)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 右図：2次近似（Fisher計量による）
ax2 = axes[1]
cs2 = ax2.contour(MU, SIGMA, QUAD_APPROX, levels=levels, colors='red')
ax2.clabel(cs2, inline=True, fontsize=8)
ax2.plot(mu0, sigma0, 'r*', markersize=15, label='Reference point')
ax2.set_xlabel('μ', fontsize=12)
ax2.set_ylabel('σ', fontsize=12)
ax2.set_title('Quadratic approximation (Fisher metric)\n(1/2)dθᵀ I dθ', fontsize=11)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 📝 学習ポイント

1. KLダイバージェンスは非対称: $D_{\text{KL}}(p||q) \neq D_{\text{KL}}(q||p)$
2. しかし**2次の近似で対称になる**:
   $$D_{\text{KL}}(p_\theta || p_{\theta+d\theta}) \approx \frac{1}{2} d\theta^\top I(\theta) d\theta$$
3. この2次形式がリーマン距離の2乗
4. 等高線の形がFisher計量楕円に対応
5. **あなたのKLダイバージェンスの経験が直接活きる！**

---
## 4. 計量による距離の計算例

### ユークリッド距離 vs Fisher距離

同じ「座標の差」でも、Fisher距離は σ に依存して変わる。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左図：同じ座標距離でもFisher距離は異なる
ax1 = axes[0]

# 経路1: σが小さい領域 (0.5 → 0.6)
mu1, sigma1_start, sigma1_end = 0, 0.5, 0.6
ax1.plot([mu1, mu1], [sigma1_start, sigma1_end], 'r-', linewidth=3,
         label=f'Path 1: σ={sigma1_start}→{sigma1_end}')
ax1.plot(mu1, sigma1_start, 'ro', markersize=10)
ax1.plot(mu1, sigma1_end, 'rs', markersize=10)

# 経路2: σが大きい領域 (2.0 → 2.1)
mu2, sigma2_start, sigma2_end = 1, 2.0, 2.1
ax1.plot([mu2, mu2], [sigma2_start, sigma2_end], 'b-', linewidth=3,
         label=f'Path 2: σ={sigma2_start}→{sigma2_end}')
ax1.plot(mu2, sigma2_start, 'bo', markersize=10)
ax1.plot(mu2, sigma2_end, 'bs', markersize=10)

# Fisher距離の計算
d_sigma = 0.1
fisher_dist_1 = np.sqrt(2) * d_sigma / sigma1_start
fisher_dist_2 = np.sqrt(2) * d_sigma / sigma2_start

ax1.text(mu1 + 0.1, (sigma1_start + sigma1_end)/2,
         f'Fisher dist≈{fisher_dist_1:.3f}', fontsize=10, color='red')
ax1.text(mu2 + 0.1, (sigma2_start + sigma2_end)/2,
         f'Fisher dist≈{fisher_dist_2:.3f}', fontsize=10, color='blue')

ax1.set_xlabel('μ', fontsize=12)
ax1.set_ylabel('σ', fontsize=12)
ax1.set_title('Same coordinate distance, different Fisher distance\nSmall σ region feels "farther"', fontsize=11)
ax1.legend()
ax1.set_xlim(-0.5, 2)
ax1.set_ylim(0, 3)
ax1.grid(True, alpha=0.3)

# 右図：対応する分布の変化
ax2 = axes[1]
x = np.linspace(-4, 6, 200)

# 経路1の分布変化
y1_start = gaussian_pdf(x, mu1, sigma1_start)
y1_end = gaussian_pdf(x, mu1, sigma1_end)
ax2.plot(x, y1_start, 'r-', linewidth=2, label=f'σ={sigma1_start}')
ax2.plot(x, y1_end, 'r--', linewidth=2, label=f'σ={sigma1_end}')

# 経路2の分布変化
y2_start = gaussian_pdf(x, mu2, sigma2_start)
y2_end = gaussian_pdf(x, mu2, sigma2_end)
ax2.plot(x, y2_start, 'b-', linewidth=2, label=f'σ={sigma2_start}')
ax2.plot(x, y2_end, 'b--', linewidth=2, label=f'σ={sigma2_end}')

ax2.set_xlabel('x', fontsize=12)
ax2.set_ylabel('p(x)', fontsize=12)
ax2.set_title('Distribution changes\nNarrow dist (red) shows larger visual change', fontsize=11)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"""
【計算結果】
- Path 1 (σ=0.5→0.6): Fisher distance ≈ {fisher_dist_1:.4f}
- Path 2 (σ=2.0→2.1): Fisher distance ≈ {fisher_dist_2:.4f}
- Ratio: {fisher_dist_1/fisher_dist_2:.2f}x
""")

### 📝 学習ポイント

1. 座標の変化量が同じでも、Fisher距離は違う
2. σが小さい領域での変化は「情報的に大きい」
3. **これがカルマンフィルタで観測精度が高いほど更新が効く理由の幾何学的説明**
4. 自然勾配法では、この計量を考慮して最適化する

---
## 5. 確認問題

### Q1. Fisher情報の計算
ポアソン分布 $\text{Poisson}(\lambda)$ のFisher情報 $I(\lambda)$ を計算せよ。

### Q2. 計量による長さ
正規分布の多様体で、点 $(\mu, \sigma) = (0, 1)$ における接ベクトル $v = (1, 1)$ のFisher長さを計算せよ。

### Q3. 距離の比較
$(0, 1) \to (0.1, 1)$ と $(0, 2) \to (0.1, 2)$ の移動で、どちらがFisher距離が大きいか？

In [ ]:
# Q1の解答
print("""
Q1 解答:
ポアソン分布: P(x|λ) = λˣ e⁻ᵏ / x!

対数尤度: log P(x|λ) = x log λ - λ - log(x!)

スコア関数:
∂ log P / ∂λ = x/λ - 1

Fisher情報:
I(λ) = E[(∂ log P / ∂λ)²]
     = E[(x/λ - 1)²]
     = E[x²/λ² - 2x/λ + 1]
     = (Var(x) + E[x]²)/λ² - 2E[x]/λ + 1
     = (λ + λ²)/λ² - 2λ/λ + 1
     = 1/λ + 1 - 2 + 1
     = 1/λ

答: I(λ) = 1/λ

（λが小さいほどFisher情報が大きい → 推定精度が高い）
""")

In [ ]:
# Q2の解答
mu, sigma = 0, 1
v = np.array([1, 1])
I = compute_fisher_gaussian(mu, sigma)

fisher_length_sq = v @ I @ v
fisher_length = np.sqrt(fisher_length_sq)

print(f"""
Q2 解答:
点 (μ, σ) = ({mu}, {sigma}) での Fisher情報行列:
I = {I}

接ベクトル v = {v}

Fisher長さの2乗:
||v||² = vᵀ I v = {v} @ {I} @ {v} = {fisher_length_sq}

Fisher長さ: ||v|| = √{fisher_length_sq} = {fisher_length:.4f}
""")

In [ ]:
# Q3の解答
# 移動1: (0, 1) → (0.1, 1)  dμ=0.1, dσ=0
d_mu1, sigma1 = 0.1, 1
dist1 = np.sqrt(d_mu1**2 / sigma1**2)  # √(dμ²/σ²)

# 移動2: (0, 2) → (0.1, 2)  dμ=0.1, dσ=0
d_mu2, sigma2 = 0.1, 2
dist2 = np.sqrt(d_mu2**2 / sigma2**2)

print(f"""
Q3 解答:
移動1: (0, 1) → (0.1, 1), dμ = 0.1, dσ = 0
Fisher距離 = √(dμ²/σ²) = √(0.1²/1²) = {dist1:.4f}

移動2: (0, 2) → (0.1, 2), dμ = 0.1, dσ = 0
Fisher距離 = √(dμ²/σ²) = √(0.1²/2²) = {dist2:.4f}

答: 移動1の方がFisher距離が大きい（{dist1/dist2:.1f}倍）

直感: σが小さい領域では、同じμの変化でも「情報的に」大きな変化
""")

---
## まとめ

| 概念 | 説明 |
|-----|------|
| Fisher情報行列 | スコア関数の共分散、リーマン計量になる |
| KLの2次近似 | Fisher計量による距離の2乗 |
| 計量楕円 | 各点での「単位球」の形 |
| 情報的距離 | σが小さい領域で大きくなる |

---
**次のステップ**: 第2章で統計的モデルの幾何学的構造を本格的に学ぶ